# Fine-tuning CamemBERT on FQuAD: Transformer head

Fine-tunes `camembert-base` with extra Transformer encoder blocks (implemented from scratch) between CamemBERT and the span-prediction layer.

Requires the FQuAD `train.json` and `valid.json` files in the working directory. Written for a GPU runtime (Google Colab).

## Training and validation data

In [ ]:
import json
from pandas import json_normalize
import pandas as pd
import numpy as np

In [ ]:
def formatting_squad(dataset_file, display=False):
    with open(dataset_file, encoding='utf-8') as f:
        raw_data = json.load(f)
        raw_data = raw_data['data']
        raw_data = json_normalize(raw_data)['paragraphs']
        context = []
        question = []
        answer_start = []
        text = []
        answers = []
        for i in range(len(raw_data)): # paragraphs
            for j in range(len(raw_data[i])): #qas
                for k in range(len(raw_data[i][j]['qas'])):
                    if (len(raw_data[i][j]['qas'][k]['answers']) != 0):
                        question.append(raw_data[i][j]['qas'][k]['question'])
                        answer_start.append(raw_data[i][j]['qas'][k]['answers'][0]['answer_start'])
                        text.append(raw_data[i][j]['qas'][k]['answers'][0]['text'])
                        answers.append([a['text'] for a in raw_data[i][j]['qas'][k]['answers']])
                        context.append(raw_data[i][j]['context'])
                    else:
                        continue
        data = pd.DataFrame({"context":context, "question": question, "answer_start": answer_start, "text": text, "answers": answers})
        if display is True:
            print(data.shape)
        return data

In [ ]:
train_data = formatting_squad(r"train.json")
train_data = train_data[train_data['context'].apply(lambda x: len(x)<500)].reset_index(drop=True)

In [ ]:
valid_data = formatting_squad(r"valid.json")
valid_data = valid_data.reset_index(drop=True)

## Defining the model

In [ ]:
!pip install transformers==2.9.1

In [ ]:
import transformers
from torch import nn
import torch
from torch.utils.data import Dataset, DataLoader
from transformers.optimization import AdamW
from tqdm import trange, tqdm_notebook

In [ ]:
from transformers.modeling_camembert import CamembertModel
from transformers.tokenization_camembert import CamembertTokenizer

### Transformer encoder

In [ ]:
from enum import IntEnum

class Dim(IntEnum):
    batch = 0
    seq = 1
    feature = 2

In [ ]:
import math

class ScaledDotProductAttention(nn.Module):
    def __init__(self, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

    def forward(self, q, k, v, mask=None):
        d_k = k.size(-1) # get the size of the key
        assert q.size(-1) == d_k

        # compute the dot product between queries and keys for
        # each batch and position in the sequence
        attn = torch.bmm(q, k.transpose(Dim.seq, Dim.feature)) # (Batch, Seq, Seq)
        # we get an attention score between each position in the sequence
        # for each batch

        # scale the dot products by the dimensionality (see the paper for why we do this!)
        attn = attn / math.sqrt(d_k)
        # normalize the weights across the sequence dimension
        # (Note that since we transposed, the sequence and feature dimensions are switched)
        attn = torch.exp(attn)
        
        # fill attention weights with 0s where padded
        if mask is not None: attn = attn.masked_fill(mask, 0)
        attn = attn / attn.sum(dim=-1, keepdim=True)
        attn = self.dropout(attn)
        output = torch.bmm(attn, v) # (Batch, Seq, Feature)
        return output

In [ ]:
class AttentionHead(nn.Module):
    def __init__(self, d_model, d_feature, dropout=0.1):
        super().__init__()
        # We will assume the queries, keys, and values all have the same feature size
        self.attn = ScaledDotProductAttention(dropout)
        self.query_tfm = nn.Linear(d_model, d_feature)
        self.key_tfm = nn.Linear(d_model, d_feature)
        self.value_tfm = nn.Linear(d_model, d_feature)

    def forward(self, queries, keys, values, mask=None):
        Q = self.query_tfm(queries) # (Batch, Seq, Feature)
        K = self.key_tfm(keys) # (Batch, Seq, Feature)
        V = self.value_tfm(values) # (Batch, Seq, Feature)
        # compute multiple attention weighted sums
        x = self.attn(Q, K, V)
        return x

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, d_feature, n_heads, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.d_feature = d_feature
        self.n_heads = n_heads
        # in practice, d_model == d_feature * n_heads
        assert d_model == d_feature * n_heads

        # Note that this is very inefficient:
        # I am merely implementing the heads separately because it is 
        # easier to understand this way
        self.attn_heads = nn.ModuleList([
            AttentionHead(d_model, d_feature, dropout) for _ in range(n_heads)
        ])
        self.projection = nn.Linear(d_feature * n_heads, d_model) 
    
    def forward(self, queries, keys, values, mask=None):
        x = [attn(queries, keys, values, mask=mask) # (Batch, Seq, Feature)
             for i, attn in enumerate(self.attn_heads)]
        
        # reconcatenate
        x = torch.cat(x, dim=Dim.feature) # (Batch, Seq, D_Feature * n_heads)
        x = self.projection(x) # (Batch, Seq, D_Model)
        return x

In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, d_model, eps=1e-8):
        super(LayerNorm, self).__init__()
        self.gamma = nn.Parameter(torch.ones(d_model))
        self.beta = nn.Parameter(torch.zeros(d_model))
        self.eps = eps

    def forward(self, x):
        mean = x.mean(-1, keepdim=True)
        std = x.std(-1, keepdim=True)
        return self.gamma * (x - mean) / (std + self.eps) + self.beta

In [ ]:
class EncoderBlock(nn.Module):
    def __init__(self, d_model=768, d_feature=64,
                 d_ff=3072, n_heads=12, dropout=0.1):
        super().__init__()
        self.attn_head = MultiHeadAttention(d_model, d_feature, n_heads, dropout)
        self.layer_norm1 = LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.position_wise_feed_forward = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model),
        )
        self.layer_norm2 = LayerNorm(d_model)
        
    def forward(self, x, mask=None):
        att = self.attn_head(x, x, x, mask=mask)
        # Apply normalization and residual connection
        x = x + self.dropout(self.layer_norm1(att))
        # Apply position-wise feedforward network
        pos = self.position_wise_feed_forward(x)
        # Apply normalization and residual connection
        x = x + self.dropout(self.layer_norm2(pos))
        return x

In [ ]:
class TransformerEncoder(nn.Module):
    def __init__(self, n_blocks=2, d_model=768
                 , n_heads=12, d_ff=3072, dropout=0.1):
        super().__init__()
        self.encoders = nn.ModuleList([
            EncoderBlock(d_model=d_model, d_feature=d_model // n_heads,
                         d_ff=d_ff, dropout=dropout)
            for _ in range(n_blocks)
        ])
    
    def forward(self, x: torch.FloatTensor, mask=None):
        for encoder in self.encoders:
            x = encoder(x)
        return x

### CamemBERT with a Transformer head

In [ ]:
class CamemBERTQA(nn.Module):
    def __init__(self,bert_type, hidden_size, num_labels, n_blocks = 2, n_heads = 12, dropout=0.1):
        super(CamemBERTQA, self).__init__()
        self.bert_type = bert_type
        self.hidden_size = hidden_size
        self.num_labels = num_labels
        self.n_blocks = n_blocks
        self.n_heads = n_heads
        self.dropout = dropout
        self.camembert = CamembertModel.from_pretrained(self.bert_type)
        self.transformer = TransformerEncoder(n_blocks = self.n_blocks, d_model = self.hidden_size
                                              , n_heads=self.n_heads, d_ff=3072, dropout=self.dropout)
        self.qa_outputs = nn.Linear(self.hidden_size, self.num_labels)

    def forward(self, input_ids, mask=None):
        output = self.camembert(input_ids = input_ids) # input_ids is a tensor
        sequence_output = self.transformer(output[0]) # Transformer
        logits = self.qa_outputs(sequence_output) #(None, seq_len, hidden_size)*(hidden_size, 2)=(None, seq_len, 2)
        start_logits, end_logits = logits.split(1, dim=-1)    #(None, seq_len, 1), (None, seq_len, 1)
        start_logits = start_logits.squeeze(-1)  #(None, seq_len)
        end_logits = end_logits.squeeze(-1)    #(None, seq_len)
        outputs = (start_logits, end_logits,) 
        return outputs

## Loss function

In [ ]:
def loss_func(out, s_target, e_target):
    criterion = nn.CrossEntropyLoss()
    s_loss = criterion(out[0], s_target)
    e_loss = criterion(out[1], e_target)
    total_loss = s_loss+e_loss
    return total_loss

## Dataloader for train and eval set

In [ ]:
class BertDatasetModule(Dataset):
    def __init__(self, tokenizer, context, question, max_length, text):
        self.context = context
        self.question = question
        self.text = text
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.context)
  
    def __getitem__(self, idx):
        context_ = self.context[idx]
        question_ = self.question[idx]
        text_ = self.text[idx]
    
        #encoding
        input_ids = self.tokenizer.encode(question_, context_)
        answer_ids = self.tokenizer.encode(text_)
        token_type_ids = [0 if i <= input_ids.index(6) else 1 for i in range(len(input_ids))]
    
        #calculating start and end position of the answer in input_ids
        s_pos, e_pos = 0, 0
        for i in range(len(input_ids)):
            if (input_ids[i: i+len(answer_ids[1:-1])] == answer_ids[1:-1]):
                s_pos = i
                e_pos = i + len(answer_ids[1:-1]) - 1
                break
        # Answer cut off by truncation: point at <s> instead of a position the model never sees,
        # as Hugging Face's SQuAD script does for answers outside the input window
        if e_pos >= self.max_length:
            s_pos, e_pos = 0, 0

        assert((s_pos<len(input_ids)) & (e_pos<len(input_ids)) & (s_pos<=e_pos))

        if (len(input_ids) < self.max_length):
            padding_len = self.max_length - len(input_ids)
            ids = input_ids + ([0]*padding_len)
        else:
            ids = input_ids[:self.max_length]

        if (len(token_type_ids)<self.max_length):
            padding_len = self.max_length - len(token_type_ids)
            token_ids = token_type_ids  + ([1]*padding_len)
        else:
            token_ids = token_type_ids[:self.max_length]

        return {'ids': torch.tensor(ids, dtype = torch.long),
                'token_type_ids': torch.tensor(token_ids, dtype = torch.long),
                'start_pos': torch.tensor(s_pos, dtype = torch.long),
                'end_pos': torch.tensor(e_pos, dtype = torch.long)}

## Training and evaluation function

In [ ]:
def train_loop(dataloader, model, optimizer, device, max_grad_norm, scheduler=None):
    model.train()
    for bi, d in enumerate(tqdm_notebook(dataloader, desc="Iteration")):
        ids = d['ids']
        token_ids = d['token_type_ids']
        start_pos = d['start_pos']
        end_pos = d['end_pos']

        ids = ids.to(device, dtype = torch.long)
        token_ids = token_ids.to(device, dtype = torch.long)
        start_pos = start_pos.to(device, dtype = torch.long)
        end_pos = end_pos.to(device, dtype = torch.long)

        optimizer.zero_grad()
        start_and_end_scores = model(ids)
        loss = loss_func(start_and_end_scores, start_pos, end_pos)
        loss.backward()
        optimizer.step()
        if scheduler is not None:
            scheduler.step()
        if bi%100==0:
            print (f"bi: {bi}, loss: {loss}")

In [ ]:
def eval_loop(dataloader, model, device):
    model.eval()
    pred_s = None
    pred_e = None
    eval_loss = 0.0
    eval_steps = 0

    for bi, d in enumerate(dataloader):
        ids = d['ids']
        token_ids = d['token_type_ids']
        start_pos = d['start_pos']
        end_pos = d['end_pos']

        ids = ids.to(device, dtype = torch.long)
        token_ids = token_ids.to(device, dtype = torch.long)
        start_pos = start_pos.to(device, dtype = torch.long)
        end_pos = end_pos.to(device, dtype = torch.long)

        with torch.no_grad():
            start_and_end_scores = model(ids)

            loss = loss_func(start_and_end_scores, start_pos, end_pos)
            eval_loss += loss.mean().item()

        eval_steps += 1
        if pred_s is None:
            pred_s = start_and_end_scores[0].detach().cpu().numpy()
            pred_e = start_and_end_scores[1].detach().cpu().numpy()
        else:
            pred_s = np.append(pred_s, start_and_end_scores[0].detach().cpu().numpy(), axis=0)
            pred_e = np.append(pred_e, start_and_end_scores[1].detach().cpu().numpy(), axis=0)

    eval_loss = eval_loss/eval_steps
    pred_start = np.argmax(pred_s, axis=1)
    pred_end = np.argmax(pred_e, axis=1)

    return eval_loss, pred_start, pred_end

## Configuration

In [ ]:
MAX_SEQ_LENGTH = 512
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 16
LEARNING_RATE = 1e-5
NUM_TRAIN_EPOCHS = 3
BERT_TYPE = "camembert-base"
max_grad_norm = 1.0

In [ ]:
tokenizer = CamembertTokenizer.from_pretrained(BERT_TYPE, do_lower_case = True)

train_dataset = BertDatasetModule(
    tokenizer = tokenizer,
    context = train_data['context'],
    question = train_data['question'],
    max_length = MAX_SEQ_LENGTH,
    text = train_data['text']
)

train_dataloader = DataLoader(train_dataset, batch_size = TRAIN_BATCH_SIZE, shuffle=True)

In [ ]:
eval_dataset = BertDatasetModule(
    tokenizer = tokenizer,
    context = valid_data['context'],
    question = valid_data['question'],
    max_length = MAX_SEQ_LENGTH,
    text = valid_data['text']
)

eval_dataloader = DataLoader(eval_dataset, batch_size = EVAL_BATCH_SIZE, shuffle=False)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

model = CamemBERTQA(bert_type = BERT_TYPE, hidden_size = 768, num_labels = 2, n_blocks = 2, n_heads = 12).to(device)
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, correct_bias=False)

NUM_TRAIN_STEPS = int(len(train_dataset)/TRAIN_BATCH_SIZE * NUM_TRAIN_EPOCHS) 
scheduler = transformers.get_constant_schedule_with_warmup(
                optimizer,
                num_warmup_steps=500,
                last_epoch=-1)

## Training iterations

In [ ]:
#training
for epoch in trange(NUM_TRAIN_EPOCHS):
    train_loop(train_dataloader, model, optimizer, device, max_grad_norm, scheduler)

## Evaluation

In [ ]:
res = eval_loop(eval_dataloader, model, device)
print(res[0])

## Testing the model

In [ ]:
dev_data = formatting_squad("valid.json")
print(dev_data.shape)
dev_data.head(3)

In [ ]:
pred_start, pred_end = res[1], res[2]
input_ids_list = [tokenizer.encode(q, c) for q, c in zip(dev_data['question'], dev_data['context'])]
dev_data['predicted_text'] = [
    tokenizer.decode(ids[start:end + 1], skip_special_tokens=True)
    for ids, start, end in zip(input_ids_list, pred_start, pred_end)
]

## Exact Match and F1

In [ ]:
import re
import string
from collections import Counter

# SQuAD v1.1 evaluation adapted to French as in the FQuAD paper
# (d'Hoffschmidt et al., 2020): ignore case, punctuation and the articles
# le, la, les, l', du, des, au, aux, un, une.
ARTICLES = re.compile(r"\b(?:le|la|les|du|des|au|aux|un|une)\b|\bl'")
PUNCTUATION = set(string.punctuation) | set("«»…“”")

def normalize_answer(text):
    text = text.lower().replace("’", "'")
    text = ARTICLES.sub(" ", text)
    text = "".join(ch for ch in text if ch not in PUNCTUATION)
    return " ".join(text.split())

def exact_match(prediction, ground_truth):
    return float(normalize_answer(prediction) == normalize_answer(ground_truth))

def f1(prediction, ground_truth):
    pred_tokens = normalize_answer(prediction).split()
    truth_tokens = normalize_answer(ground_truth).split()
    num_same = sum((Counter(pred_tokens) & Counter(truth_tokens)).values())
    if num_same == 0:
        return 0.0
    precision = num_same / len(pred_tokens)
    recall = num_same / len(truth_tokens)
    return 2 * precision * recall / (precision + recall)

def best_score(metric, prediction, ground_truths):
    """Questions can have several gold answers: keep the best match."""
    return max(metric(prediction, truth) for truth in ground_truths)

In [ ]:
dev_data['exact_match'] = [best_score(exact_match, p, a) for p, a in zip(dev_data['predicted_text'], dev_data['answers'])]
dev_data['f1'] = [best_score(f1, p, a) for p, a in zip(dev_data['predicted_text'], dev_data['answers'])]

print(f"Exact Match = {100 * dev_data['exact_match'].mean():.1f}")
print(f"F1 = {100 * dev_data['f1'].mean():.1f}")

In [ ]:
show_columns = ['question', 'answers', 'predicted_text', 'exact_match', 'f1']
dev_data[show_columns].head(20)

In [ ]:
errors = dev_data[dev_data['exact_match'] == 0]
errors[show_columns].head()

In [ ]:
errors[show_columns].to_csv('csv_incorrect.csv')
dev_data.drop(errors.index)[show_columns].to_csv('csv_correct.csv')